# 624 Hektor

## Initialize

In [19]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.time import Time
from astroquery.jplhorizons import Horizons

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "font.size": 9,
    "axes.labelsize": 9,
    "legend.fontsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
})

In [20]:
DAYS_PER_JULIAN_YEAR = 365.25

# primary body (sun)
NAME_PRIMARY = 'Sun'
COLOR_PRIMARY = 'darkorange'

# secondary body (jupiter)
NAME_SECONDARY = 'Jupiter'
COLOR_SECONDARY = 'chocolate'

# particle (624 Hektor)
NAME_PARTICLE = '624 Hektor'
COLOR_PARTICLE = 'lightseagreen'

EPOCH_START_JD = 2461143.5

In [21]:
def get_rotation_coords(vec_particle, vec_secondary):
    theta = np.arctan2(vec_secondary['y'], vec_secondary['x'])
    omega = (vec_secondary['x'] * vec_secondary['vy'] - vec_secondary['y'] * vec_secondary['vx']) / (vec_secondary['x']**2 + vec_secondary['y']**2)

    dx = vec_particle['x'] - vec_secondary['x']
    dy = vec_particle['y'] - vec_secondary['y']
    dz = vec_particle['z'] - vec_secondary['z']
    
    dvx = vec_particle['vx'] - vec_secondary['vx']
    dvy = vec_particle['vy'] - vec_secondary['vy']
    dvz = vec_particle['vz'] - vec_secondary['vz']
    
    x_rot = dx * np.cos(theta) + dy * np.sin(theta)
    y_rot = -dx * np.sin(theta) + dy * np.cos(theta)
    z_rot = dz
    
    vx_rot = (dvx * np.cos(theta) + dvy * np.sin(theta)) + (omega * y_rot)
    vy_rot = (-dvx * np.sin(theta) + dvy * np.cos(theta)) - (omega * x_rot)
    vz_rot = dvz
    
    return pd.DataFrame({
        'x': x_rot, 'y': y_rot, 'z': z_rot,
        'vx': vx_rot, 'vy': vy_rot, 'vz': vz_rot
    })

In [22]:
# Mercury6 data
col = ['time', 'long', 'x', 'y', 'z', 'vx', 'vy', 'vz']

vec_jupiter = pd.read_csv('JUPITER.aei', sep='\s+', skiprows=4, names=col)
x_jupiter, y_jupiter, z_jupiter = vec_jupiter['x'], vec_jupiter['y'], vec_jupiter['z']

vec_hektor = pd.read_csv('HEKTOR.aei', sep='\s+', skiprows=4, names=col)
vec_hektor_rot = get_rotation_coords(vec_hektor, vec_jupiter)
x_icrf, y_icrf, z_icrf = vec_hektor['x'], vec_hektor['y'], vec_hektor['z']
x_rot, y_rot, z_rot = vec_hektor_rot['x'], vec_hektor_rot['y'], vec_hektor_rot['z']

In [ ]:
# Horizons Data
duration_years = 10
stop_jd = EPOCH_START_JD + (duration_years * DAYS_PER_JULIAN_YEAR)

start_iso = Time(EPOCH_START_JD, format='jd').to_value('iso', subfmt='date')
stop_iso = Time(stop_jd, format='jd').to_value('iso', subfmt='date')

epochs = {
    'start': start_iso,
    'stop': stop_iso,
    'step': '60d'
}

obj_jupiter_hor = Horizons(id='5', location='@sun', epochs=epochs)
vec_jupiter_hor = obj_jupiter_hor.vectors()

obj_hektor_hor = Horizons(id='Hektor', location='@sun', epochs=epochs)
vec_hektor_hor = obj_hektor_hor.vectors()

vec_hektor_hor_rot = get_rotation_coords(vec_hektor_hor, vec_jupiter_hor)
x_rot_hor, y_rot_hor, z_rot_hor = vec_hektor_hor_rot['x'], vec_hektor_hor_rot['y'], vec_hektor_hor_rot['z']

## ICRF

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))

ax.plot(x_icrf, y_icrf, label=NAME_PARTICLE, color=COLOR_PARTICLE, alpha=0.7, zorder=2)
ax.plot(x_jupiter, y_jupiter, label=NAME_SECONDARY, color=COLOR_SECONDARY, alpha=0.7, zorder=3)
ax.scatter(0, 0, label=NAME_PRIMARY, color=COLOR_PRIMARY, s=40, zorder=2)

ax.set_xlim(-6, 6)
ax.set_ylim(-6, 6)
ax.set_xlabel('$x$ (au)')
ax.set_ylabel('$y$ (au)')
ax.legend()
ax.grid(True, linestyle=':')

fig.savefig('hektor_icrf.pdf', format='pdf', bbox_inches='tight')
plt.show()

## Jupiter-centered Rotating Frame

In [ ]:
# L4 Calculation
d_j = np.sqrt(vec_jupiter['x']**2 + vec_jupiter['y']**2 + vec_jupiter['z']**2)
l4_x = -0.5 * d_j
l4_y = (np.sqrt(3) / 2) * d_j
l4_z = 0 * d_j

l4_dist = np.sqrt((vec_hektor_rot['x'] - l4_x)**2 + (vec_hektor_rot['y'] - l4_y)**2 + (vec_hektor_rot['z'] - l4_z)**2)

print(f"Maximum distance from L4 point: {l4_dist.max():.3f} au")

# Plotting
fig = plt.figure(figsize=(6, 6))
gs = fig.add_gridspec(2, 2, 
                      width_ratios=[3, 2],
                      height_ratios=[3, 2])

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[1, 0], sharex=ax1)
ax3 = fig.add_subplot(gs[0, 1], sharey=ax1)

ax1.plot(x_rot, y_rot, label='Simulated Trajectory', color=COLOR_PARTICLE, alpha=0.3, zorder=2)
ax1.scatter(x_rot_hor, y_rot_hor, label='Horizons (10 years)', color=COLOR_PARTICLE, marker='+', s=40, zorder=3)
ax1.plot(l4_x, l4_y, color='darkred', label='Sun--Jupiter $L_4$', zorder=3)
ax1.scatter(0, 0, label=NAME_SECONDARY, color=COLOR_SECONDARY, s=40, zorder=3)

ax1.set_aspect('equal')
ax1.set_ylabel('$y$ (au)')
ax1.set_xlim(-5.5, 0.5)
ax1.set_ylim(-0.5, 5.5)
ax1.grid(True, linestyle=':')

ax2.plot(x_rot, z_rot, color=COLOR_PARTICLE, alpha=0.3, zorder=2)
ax2.scatter(x_rot_hor, z_rot_hor, color=COLOR_PARTICLE, marker='+', s=40, zorder=3)
ax2.plot(l4_x, l4_z, color='darkred',zorder=3)
ax2.scatter(0, 0, color=COLOR_SECONDARY, s=40, zorder=3)

ax2.set_aspect('equal')
ax2.set_xlabel('$x$ (au)')
ax2.set_ylabel('$z$ (au)')
ax2.set_ylim(-2, 2)
ax2.grid(True, linestyle=':')

ax3.plot(z_rot, y_rot, color=COLOR_PARTICLE, alpha=0.3, zorder=2)
ax3.scatter(z_rot_hor, y_rot_hor, color=COLOR_PARTICLE, marker='+', s=40, zorder=3)
ax3.plot(l4_z, l4_y, color='darkred', zorder=3)
ax3.scatter(0, 0, color=COLOR_SECONDARY, s=40, zorder=3)

ax3.set_aspect('equal')
ax3.set_xlabel('$z$ (au)')
ax3.set_ylim(-0.5, 5.5)
ax3.grid(True, linestyle=':')

ax_legend = fig.add_subplot(gs[1, 1])
ax_legend.axis('off')
handles, labels = ax1.get_legend_handles_labels()
ax_legend.legend(handles, labels, loc='center')

fig.align_ylabels()
fig.savefig('hektor_rot.pdf', format='pdf', bbox_inches='tight')
plt.show()